# Задание 8. Сравнение алгоритмов классификации SMS, подбор порога и проверка многоязычности

**Цель:** сравнить 4 алгоритма классификации (`MultinomialNB`, `LogisticRegression`, `LinearSVC + CalibratedClassifierCV`, `RandomForestClassifier`) и 4 варианта формирования признаков на едином разбиении (`StratifiedKFold(n_splits=5)` на `train`), подобрать оптимальный порог классификации на `val`, однократно оценить итоговую модель на `test`, переобучить на `train + val` и проверить качество на русско-казахском корпусе `data/raw/custom_ru_kk.csv`.


In [1]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with (PROJECT_ROOT / "ml/artifacts/sms_model_meta.json").open(encoding="utf-8") as f:
    meta = json.load(f)

df_feat = pd.DataFrame(meta["feature_variants_comparison_val"])[
    ["variant_code", "n_features", "accuracy", "precision_spam", "recall_spam", "f1_spam", "f2_spam", "roc_auc", "pr_auc"]
]
print("=== Сравнение 4 вариантов признаков на валидации ===")
print(df_feat.to_string(index=False))


=== Сравнение 4 вариантов признаков на валидации ===
                         variant_code  n_features  accuracy  precision_spam  recall_spam  f1_spam  f2_spam  roc_auc  pr_auc
              words_only_no_stopwords        8555    0.9791          0.8776       0.9556   0.9149   0.9389   0.9906  0.9285
            words_only_with_stopwords        4757    0.9647          0.7890       0.9556   0.8643   0.9168   0.9851  0.9128
  words_plus_char_manual_no_stopwords       28479    0.9830          0.9140       0.9444   0.9290   0.9382   0.9935  0.9734
words_plus_char_manual_with_stopwords       24553    0.9791          0.8854       0.9444   0.9140   0.9320   0.9919  0.9687


In [2]:
df_models = pd.DataFrame(meta["models_comparison_val"])[
    ["model_name", "best_params", "accuracy", "precision_spam", "recall_spam", "f1_spam", "f2_spam", "roc_auc", "pr_auc", "train_time_sec", "single_sms_latency_ms"]
]
print("=== Сводная таблица сравнения 4 алгоритмов на валидации (порог 0.50) ===")
print(df_models.to_string(index=False))


=== Сводная таблица сравнения 4 алгоритмов на валидации (порог 0.50) ===
                         model_name                                        best_params  accuracy  precision_spam  recall_spam  f1_spam  f2_spam  roc_auc  pr_auc  train_time_sec  single_sms_latency_ms
            Multinomial Naive Bayes                                {'clf__alpha': 0.1}    0.9856          0.9247       0.9556   0.9399   0.9492   0.9982  0.9883           0.912                  2.143
     Logistic Regression (balanced)                                    {'clf__C': 5.0}    0.9856          0.9341       0.9444   0.9392   0.9424   0.9937  0.9768           1.070                  1.728
Linear SVM + CalibratedClassifierCV                         {'clf__estimator__C': 2.0}    0.9882          0.9551       0.9444   0.9497   0.9465   0.9946  0.9800           0.650                  3.228
           Random Forest (balanced) {'clf__max_depth': None, 'clf__n_estimators': 200}    0.9856          0.9540       0.9222  

In [3]:
print("Выбранная модель:", meta["algorithm"], meta["hyperparameters"])
print("Подобранный на валидации порог:", meta["threshold"])
print("\nМетрики на валидации при пороге 0.53:", json.dumps(meta["val_metrics"], indent=2))
print("\nОднократная оценка на отложенной тестовой выборке (test, N=765):", json.dumps(meta["test_metrics"], indent=2))


Выбранная модель: Logistic Regression (balanced) {'clf__C': 5.0}
Подобранный на валидации порог: 0.53

Метрики на валидации при пороге 0.53: {
  "threshold": 0.53,
  "accuracy": 0.9882,
  "precision_spam": 0.9551,
  "recall_spam": 0.9444,
  "f1_spam": 0.9497,
  "f2_spam": 0.9465,
  "precision_macro": 0.9738,
  "recall_macro": 0.9693,
  "f1_macro": 0.9715,
  "roc_auc": 0.9937,
  "pr_auc": 0.9768,
  "confusion_matrix": {
    "tn": 671,
    "fp": 4,
    "fn": 5,
    "tp": 85,
    "matrix": [
      [
        671,
        4
      ],
      [
        5,
        85
      ]
    ]
  }
}

Однократная оценка на отложенной тестовой выборке (test, N=765): {
  "threshold": 0.53,
  "accuracy": 0.9856,
  "precision_spam": 0.954,
  "recall_spam": 0.9222,
  "f1_spam": 0.9379,
  "f2_spam": 0.9284,
  "precision_macro": 0.9718,
  "recall_macro": 0.9581,
  "f1_macro": 0.9649,
  "roc_auc": 0.998,
  "pr_auc": 0.9878,
  "confusion_matrix": {
    "tn": 671,
    "fp": 4,
    "fn": 7,
    "tp": 83,
    "matrix": [

In [4]:
print("=== Проверка на собственном русско-казахском наборе (data/raw/custom_ru_kk.csv) ===")
for lang_key, info in meta["multilingual_evaluation"].items():
    print(f"\nЯзык: {lang_key} (N={info['count']}, spam={info['spam_count']}, ham={info['ham_count']})")
    print("  1) Чистая англоязычная ML-модель:", info["pure_ml_model"])
    print("  2) Гибридный контур (ML + словарь правил СИ и наличие URL):", info["hybrid_ml_plus_rules"])


=== Проверка на собственном русско-казахском наборе (data/raw/custom_ru_kk.csv) ===

Язык: ru (N=120, spam=60, ham=60)
  1) Чистая англоязычная ML-модель: {'threshold': 0.53, 'accuracy': 0.625, 'precision_spam': 0.6027, 'recall_spam': 0.7333, 'f1_spam': 0.6617, 'f2_spam': 0.7029, 'precision_macro': 0.6312, 'recall_macro': 0.625, 'f1_macro': 0.6205, 'roc_auc': 0.6908, 'pr_auc': 0.729, 'confusion_matrix': {'tn': 31, 'fp': 29, 'fn': 16, 'tp': 44, 'matrix': [[31, 29], [16, 44]]}}
  2) Гибридный контур (ML + словарь правил СИ и наличие URL): {'threshold': 0.5, 'accuracy': 0.6833, 'precision_spam': 0.6375, 'recall_spam': 0.85, 'f1_spam': 0.7286, 'f2_spam': 0.7969, 'precision_macro': 0.7063, 'recall_macro': 0.6833, 'f1_macro': 0.6743, 'roc_auc': 0.84, 'pr_auc': 0.8796, 'confusion_matrix': {'tn': 31, 'fp': 29, 'fn': 9, 'tp': 51, 'matrix': [[31, 29], [9, 51]]}}

Язык: kk (N=60, spam=30, ham=30)
  1) Чистая англоязычная ML-модель: {'threshold': 0.53, 'accuracy': 0.6, 'precision_spam': 0.5789, 'r